# RF Camera Detection - Exploratory Data Analysis

This notebook explores the UNSW IoT Traffic dataset for camera detection.

**Objectives:**
1. Understand the dataset structure
2. Analyze class balance (camera vs non-camera)
3. Visualize flow statistics distributions
4. Identify distinguishing features between camera and non-camera traffic

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from data_loader import load_config, set_random_seed, UNSWIoTLoader, check_data_availability
from preprocess import FlowPreprocessor
from features import FlowFeatureExtractor

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Load config
config = load_config('../configs/config.yaml')
set_random_seed(config['random_seed'])

print("Configuration loaded successfully!")

## 1. Check Data Availability

In [ ]:
# Check what data is available
status = check_data_availability('../configs/config.yaml')

for dataset, info in status.items():
    symbol = '✓' if info['available'] else '✗'
    print(f"[{symbol}] {dataset}")
    if not info['available']:
        print(f"    {info['message']}")

## 2. Load UNSW IoT Dataset

In [ ]:
# Initialize loader
data_dir = Path('../') / config['paths']['data_raw']
loader = UNSWIoTLoader(data_dir, config)

# Discover files
files = loader.discover_files()
print(f"\nFlow CSV files found: {len(files['flow_csvs'])}")
print(f"Device list files found: {len(files['device_lists'])}")

In [ ]:
# Load device mapping and identify camera MACs
device_mapping = loader.load_device_mapping()
camera_macs = loader.identify_camera_macs()

print(f"\nTotal devices: {len(device_mapping)}")
print(f"Camera devices: {len(camera_macs)}")

In [ ]:
# Load flow data (use max_files for faster loading during EDA)
# Set to None to load all files
MAX_FILES = 10  # Adjust based on your system

df = loader.load_flow_csvs(max_files=MAX_FILES)
print(f"\nLoaded {len(df):,} flow records")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Create labels
df = loader.create_labels()

# Basic info
print(f"\nDataset shape: {df.shape}")
print(f"\nData types:")
print(df.dtypes)

## 3. Class Balance Analysis

In [ ]:
# Class distribution
class_counts = df['is_camera'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot
ax1 = axes[0]
bars = ax1.bar(['Non-Camera', 'Camera'], class_counts.values, color=['steelblue', 'coral'])
ax1.set_ylabel('Number of Flows')
ax1.set_title('Class Distribution')
for bar, count in zip(bars, class_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000, 
             f'{count:,}', ha='center', va='bottom', fontsize=11)

# Pie chart
ax2 = axes[1]
ax2.pie(class_counts.values, labels=['Non-Camera', 'Camera'], 
        autopct='%1.1f%%', colors=['steelblue', 'coral'], explode=[0, 0.05])
ax2.set_title('Class Proportions')

plt.tight_layout()
plt.savefig('../outputs/plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nClass imbalance ratio: 1:{class_counts[0]/class_counts[1]:.1f}")

## 4. Feature Analysis

In [ ]:
# Extract features
extractor = FlowFeatureExtractor(config.get('track_a', {}).get('features', {}))
features_df = extractor.extract_basic_features(df)
features_df['is_camera'] = df['is_camera'].values

print(f"Extracted {len(extractor.feature_names)} features")
print(f"\nFeatures: {extractor.feature_names}")

In [ ]:
# Summary statistics by class
numeric_features = [c for c in features_df.columns if c.startswith('feat_')]

print("\nSummary Statistics - Camera vs Non-Camera:")
print("="*60)

for feat in numeric_features[:10]:  # Show first 10
    camera_mean = features_df[features_df['is_camera']==1][feat].mean()
    non_camera_mean = features_df[features_df['is_camera']==0][feat].mean()
    ratio = camera_mean / (non_camera_mean + 1e-8)
    print(f"{feat:40} Camera/Non-Camera ratio: {ratio:.2f}")

In [ ]:
# Distribution plots for key features
key_features = [f for f in numeric_features if any(k in f.lower() 
                for k in ['byte', 'pkt', 'duration', 'per_sec'])][:6]

if key_features:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, feat in enumerate(key_features):
        ax = axes[idx]
        
        # Separate by class
        camera_data = features_df[features_df['is_camera']==1][feat].dropna()
        non_camera_data = features_df[features_df['is_camera']==0][feat].dropna()
        
        # Use log scale if values span many orders of magnitude
        if camera_data.max() / (camera_data.min() + 1e-8) > 100:
            camera_data = np.log1p(camera_data)
            non_camera_data = np.log1p(non_camera_data)
            feat_name = f"log({feat})"
        else:
            feat_name = feat
        
        # Plot distributions
        ax.hist(non_camera_data, bins=50, alpha=0.5, label='Non-Camera', color='steelblue', density=True)
        ax.hist(camera_data, bins=50, alpha=0.5, label='Camera', color='coral', density=True)
        ax.set_xlabel(feat_name.replace('feat_', ''))
        ax.set_ylabel('Density')
        ax.legend()

    plt.suptitle('Feature Distributions: Camera vs Non-Camera', fontsize=14)
    plt.tight_layout()
    plt.savefig('../outputs/plots/feature_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Box plots for key features
if key_features:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, feat in enumerate(key_features):
        ax = axes[idx]
        
        # Prepare data
        plot_data = features_df[[feat, 'is_camera']].copy()
        plot_data[feat] = np.log1p(plot_data[feat])  # Log transform
        plot_data['Class'] = plot_data['is_camera'].map({0: 'Non-Camera', 1: 'Camera'})
        
        sns.boxplot(x='Class', y=feat, data=plot_data, ax=ax, palette=['steelblue', 'coral'])
        ax.set_ylabel(f"log({feat.replace('feat_', '')})")
        ax.set_xlabel('')

    plt.suptitle('Feature Box Plots: Camera vs Non-Camera (log scale)', fontsize=14)
    plt.tight_layout()
    plt.savefig('../outputs/plots/feature_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Correlation Analysis

In [ ]:
# Correlation with target
correlations = features_df[numeric_features + ['is_camera']].corr()['is_camera'].drop('is_camera')
correlations = correlations.sort_values(key=abs, ascending=False)

print("Feature Correlations with Camera Label:")
print("="*50)
for feat, corr in correlations.head(15).items():
    print(f"{feat.replace('feat_', ''):40} {corr:+.4f}")

In [ ]:
# Correlation heatmap (top features)
top_features = correlations.head(10).index.tolist()

if len(top_features) > 3:
    corr_matrix = features_df[top_features + ['is_camera']].corr()
    
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=0.5)
    plt.title('Feature Correlation Heatmap (Top Features)')
    plt.tight_layout()
    plt.savefig('../outputs/plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Protocol and Port Analysis

In [ ]:
# Protocol distribution
proto_cols = [c for c in features_df.columns if 'proto' in c.lower()]

if proto_cols:
    fig, axes = plt.subplots(1, len(proto_cols), figsize=(5*len(proto_cols), 5))
    if len(proto_cols) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, proto_cols):
        data = features_df.groupby('is_camera')[col].mean()
        ax.bar(['Non-Camera', 'Camera'], data.values, color=['steelblue', 'coral'])
        ax.set_ylabel('Proportion')
        ax.set_title(col.replace('feat_', ''))
    
    plt.tight_layout()
    plt.savefig('../outputs/plots/protocol_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Port analysis
port_cols = [c for c in df.columns if 'port' in c.lower()][:2]  # src and dst

if port_cols:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for ax, col in zip(axes, port_cols):
        # Top ports for each class
        camera_ports = df[df['is_camera']==1][col].value_counts().head(10)
        non_camera_ports = df[df['is_camera']==0][col].value_counts().head(10)
        
        x = np.arange(10)
        width = 0.35
        
        ax.bar(x - width/2, camera_ports.values[:10], width, label='Camera', color='coral')
        ax.bar(x + width/2, non_camera_ports.values[:10], width, label='Non-Camera', color='steelblue')
        ax.set_xlabel('Port')
        ax.set_ylabel('Frequency')
        ax.set_title(f'Top 10 {col}')
        ax.set_xticks(x)
        ax.set_xticklabels(camera_ports.index[:10], rotation=45)
        ax.legend()
    
    plt.tight_layout()
    plt.savefig('../outputs/plots/port_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Summary Statistics

In [ ]:
# Generate summary
print("\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Total flows: {len(df):,}")
print(f"Camera flows: {df['is_camera'].sum():,} ({100*df['is_camera'].mean():.2f}%)")
print(f"Non-camera flows: {len(df) - df['is_camera'].sum():,}")
print(f"Total features extracted: {len(numeric_features)}")
print(f"\nMost correlated features:")
for feat, corr in correlations.head(5).items():
    print(f"  - {feat.replace('feat_', '')}: {corr:+.4f}")
print("="*60)

In [ ]:
# Save summary to file
summary = {
    'total_flows': len(df),
    'camera_flows': int(df['is_camera'].sum()),
    'camera_percentage': float(100*df['is_camera'].mean()),
    'num_features': len(numeric_features),
    'top_correlated_features': correlations.head(10).to_dict()
}

import json
with open('../outputs/results/eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Summary saved to outputs/results/eda_summary.json")

## 8. Key Findings

### Class Imbalance
- The dataset shows significant class imbalance between camera and non-camera traffic
- Will need to use class weighting or resampling during training

### Distinguishing Features
- Camera traffic shows distinct patterns in byte/packet counts and rates
- Video streaming creates characteristic traffic profiles
- Port and protocol distributions differ between classes

### Next Steps
1. Run `python src/train.py --config configs/config.yaml` to train models
2. Compare baseline models (LogReg, RF, XGBoost)
3. Analyze misclassified samples for feature engineering ideas